In [1]:
import pandas as pd
import json
import time
from kafka import KafkaProducer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import hour, dayofweek

In [8]:
# Load and sort the CSV by BoardingTime
df = pd.read_csv("input_data/boarding_m2.csv", parse_dates=["BoardingTime"])
df = df.sort_values("BoardingTime")

In [9]:
# Kafka setup
producer = KafkaProducer(
    bootstrap_servers='ed-kafka:29092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

topic = 'boarding-data'

In [ ]:
# Stream row by row
for _, row in df.iterrows():
    record = row.to_dict()
    #print(f"Sending: {record}")
    # Convert Timestamp to string
    if isinstance(record["BoardingTime"], pd.Timestamp):
        record["BoardingTime"] = record["BoardingTime"].isoformat()
    producer.send(topic, value=record)
    producer.flush()
    time.sleep(0.1)  # simulate 100 milliseconds intervals between boardings

print("Finished sending data.")
producer.close()